# Model Context Protocol (MCP)

MCP is a standard way for AI applications to discover and use external tools and context providers without writing a unique integration for each client-server pair.


## What to learn

- A host is the AI application the user interacts with.
- A client maintains a connection to one MCP server.
- A server exposes tools, resources, and reusable prompts.
- Capability negotiation tells each side which features are supported.

## Decision rule

Use MCP when an integration should work across multiple compatible hosts. Direct function calling is often simpler for a small, private integration. Treat server output as untrusted and enforce authorization at execution time.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""The MCP lifecycle in miniature: initialize -> discover -> call,
plus a sampling request flowing back client-ward. Message shapes mirror
the real JSON-RPC protocol.
"""
# Import the dependencies used by this example.
import json


# Define the data shape and small operations before running them.
class ToyMCPServer:
    """A 'hands' server: tools + one sampling request, no LLM key of its own."""
    def handle(self, request, client):
        method = request["method"]
        if method == "initialize":
            return {"protocolVersion": "2025-11-25",
                    "capabilities": {"tools": {"listChanged": True}}}
        if method == "tools/list":
            return {"tools": [{
                "name": "classify_ticket",
                "description": "Classify a support ticket into a queue.",
                "inputSchema": {"type": "object",
                                "properties": {"text": {"type": "string"}},
                                "required": ["text"]},
            }]}
        if method == "tools/call":
            text = request["params"]["arguments"]["text"]
            # Server needs judgment -> SAMPLING: borrow the client's model.
            label = client.sampling(f"Classify into billing/tech/other: {text}")
            return {"content": [{"type": "text", "text": f"queue={label}"}]}
        return {"error": f"unknown method {method}"}


class ToyMCPClient:
    def __init__(self, server):
        self.server = server

    def sampling(self, prompt):
        print(f"  [sampling] server asked client's LLM: {prompt[:48]}...")
        return "billing"  # the client's model answers; server stays keyless

    def call(self, method, params=None):
        return self.server.handle({"method": method, "params": params or {}}, self)


client = ToyMCPClient(ToyMCPServer())

init = client.call("initialize")
print("initialize ->", json.dumps(init))

tools = client.call("tools/list")["tools"]
print("discovered  ->", [t["name"] for t in tools])   # discovery, not hardcoding

result = client.call("tools/call", {"name": "classify_ticket",
                                    "arguments": {"text": "I was double charged"}})
print("tools/call  ->", result["content"][0]["text"])


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
